# Neural Local Volatility v7 — final NLV benchmark

This notebook freezes the Neural-LV specification selected in v6:

- **raw-price objective** (spot-normalised within each date; same optimum as raw-price MSE),
- **shared 4×50 ReLU trunk + maturity-specific Softplus heads**,
- **Sabaté-style learned control variates**,
- **deterministic 80/20 strike hold-out on every date**,
- corrected fixed-contract spot delta with common random numbers,
- correct absolute model clock for future-date pricing.

The March 1 20% set is treated as **validation**, because it was used in v6 to choose the loss and architecture. Clean test evidence comes from later dates after the specification is locked.

The main experiment is rolling calibration: load the completed v6 March 1 raw-loss maturity-head checkpoint, re-evaluate it with 100k paths, then warm-start each later date from the previous date. The trunk is copied directly and maturity heads are shifted by elapsed calendar time. The control-variate hedge bank is reinitialised each day.

Rolling updates use a **fixed low learning rate and training-only early stopping**. No daily hold-out observation is used for optimisation, checkpoint selection, or early stopping.

`COMPUTE_PROFILE="cpu"` runs five consecutive transitions. `"gpu_hpc"` uses the same scientific configuration but expands to the full available month. Classical BS/LV/Heston comparison is intentionally deferred to the final cross-model benchmark notebook after NSLV.


## 1. Setup and locked v7 configuration


In [1]:
from pathlib import Path
from dataclasses import dataclass
import copy, gc, json, math, os, random, warnings

# v5/v6 had native-kernel instability on this laptop, so CPU mode stays conservative.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq
from scipy.special import ndtr
from tqdm import tqdm
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 12345
ANCHOR_DATE = pd.Timestamp("2024-03-01")
COMPUTE_PROFILE = "cpu"          # "cpu" or "gpu_hpc"
FORCE_RETRAIN = False
RUN_SEED_ROBUSTNESS = False      # optional final section; deliberately off by default

HELDOUT_FRACTION = 0.20
MAX_QUOTES_PER_SLICE = 25
BASE_STEPS_PER_YEAR = 96

# Locked Neural-LV architecture.
SDE_HIDDEN_LAYERS, SDE_WIDTH = 4, 50
HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH = 3, 30

# Rolling calibration. Fixed LR: no scheduler in v7.
ROLLING_EPOCHS = 120
SDE_LR = 1e-4
HEDGE_LR = 1e-3
PATHS_PER_STEP = 4_096
STEPS_PER_EPOCH = 10
QUOTES_PER_MATURITY_PER_STEP = 12

# Training-only monitor / checkpoint selection / early stopping.
MONITOR_PATHS = 16_384
EVAL_EVERY_SDE = 5              # every 5 SDE phases ~= every 10 nominal epochs
TOP_K_CHECKPOINTS = 3
RERANK_PATHS = 50_000
SAVE_EVERY_EPOCHS = 10
EARLY_MIN_EPOCHS = 40
EARLY_PATIENCE = 4
EARLY_REL_DELTA = 0.002         # 0.2% relative improvement resets patience

# Final evaluation.
FINAL_MC_PATHS = 100_000
FINAL_PATH_CHUNK = 10_000
DELTA_BUMP_BPS = (1, 5, 10)
DELTA_HEADLINE_BPS = 5

# Same numerical specification under both profiles; GPU/HPC expands experiment breadth.
MAX_TRANSITIONS = 5 if COMPUTE_PROFILE == "cpu" else None

CACHE_DIR = Path("neural_lv_v7_cache")
V6_CACHE_DIR = Path("neural_lv_v6_cache")
V5_CACHE_DIR = Path("neural_lv_v5_cache")
CACHE_DIR.mkdir(exist_ok=True)

def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch, "xpu"):
        try:
            if torch.xpu.is_available():
                return torch.device("xpu")
        except Exception:
            pass
    return torch.device("cpu")

DEVICE = choose_device()
if DEVICE.type == "cpu":
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

def set_seed(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

set_seed(SEED)

print("Device:", DEVICE, "| PyTorch threads:", torch.get_num_threads())
print("Compute profile:", COMPUTE_PROFILE, "| transitions:", MAX_TRANSITIONS or "all available")
print("Locked model: raw loss + shared trunk / maturity heads")
print("Rolling: max", ROLLING_EPOCHS, "epochs | SDE LR", SDE_LR, "| hedge LR", HEDGE_LR)
print("Early stopping:", EARLY_PATIENCE, "checks after", EARLY_MIN_EPOCHS, "epochs")


Device: cpu | PyTorch threads: 1
Compute profile: cpu | transitions: 5
Locked model: raw loss + shared trunk / maturity heads
Rolling: max 120 epochs | SDE LR 0.0001 | hedge LR 0.001
Early stopping: 4 checks after 40 epochs


## 2. Load and clean OptionMetrics data


In [2]:
def find_file(pattern):
    search_dirs = [Path("."), Path("/mnt/data")]
    found = []
    for folder in search_dirs:
        found.extend(sorted(folder.glob(pattern)))
    if not found:
        raise FileNotFoundError(f"Could not find a file matching {pattern!r}")
    return found[0]


FILES = {
    "options": find_file("one_month_spx_option_price.csv"),
    "forwards": find_file("one_month_spx_forward_price.csv"),
    "spot": find_file("one_month_spx_security_price.csv"),
    "rates": find_file("one_month_spx_zero_coupon_yield_curve.csv"),
}

for name, path in FILES.items():
    print(f"{name:8s}: {path}")

options_raw = pd.read_csv(FILES["options"], low_memory=False)
forwards_raw = pd.read_csv(FILES["forwards"], low_memory=False)
spot_raw = pd.read_csv(FILES["spot"], low_memory=False)
rates_raw = pd.read_csv(FILES["rates"], low_memory=False)



SECID = 108105
MIN_DTE, MAX_DTE = 7, 365
MAX_ABS_LOG_MONEYNESS = 0.30
MIN_ABS_DELTA, MAX_ABS_DELTA = 0.05, 0.95
MAX_RELATIVE_SPREAD = 0.25
ATM_TIGHTER_SIDE_BAND = 0.05
MIN_POINTS_PER_SLICE = 8


def prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw):
    options = options_raw.copy()
    options["date"] = pd.to_datetime(options["date"])
    options["exdate"] = pd.to_datetime(options["exdate"])
    options["K"] = options["strike_price"] / 1000.0
    options["T_days"] = (options["exdate"] - options["date"]).dt.days
    options["T"] = options["T_days"] / 365.0
    options["bid"] = pd.to_numeric(options["best_bid"], errors="coerce")
    options["ask"] = pd.to_numeric(options["best_offer"], errors="coerce")
    options["mid"] = 0.5 * (options["bid"] + options["ask"])
    options["spread"] = options["ask"] - options["bid"]
    options["relative_spread"] = options["spread"] / options["mid"].replace(0, np.nan)
    options["iv_market"] = pd.to_numeric(options["impl_volatility"], errors="coerce")
    options["delta_market"] = pd.to_numeric(options["delta"], errors="coerce")
    options["abs_delta_market"] = options["delta_market"].abs()

    mask = (
        (options["secid"] == SECID)
        & (options["exercise_style"] == "E")
        & (options["contract_size"] == 100)
        & (options["cfadj"] == 1)
        & options["cp_flag"].isin(["C", "P"])
        & (options["bid"] >= 0)
        & (options["ask"] > options["bid"])
        & (options["mid"] > 0)
        & options["T_days"].between(MIN_DTE, MAX_DTE)
        & options["iv_market"].between(1e-4, 5.0)
        & options["abs_delta_market"].between(MIN_ABS_DELTA, MAX_ABS_DELTA)
        & (options["relative_spread"] <= MAX_RELATIVE_SPREAD)
    )
    options = options.loc[mask].copy()

    spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
    spot["date"] = pd.to_datetime(spot["date"])
    spot = spot.rename(columns={"close": "S"}).drop_duplicates("date")

    forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
    forwards["date"] = pd.to_datetime(forwards["date"])
    forwards["expiration"] = pd.to_datetime(forwards["expiration"])
    forwards = forwards.rename(columns={
        "expiration": "exdate",
        "AMSettlement": "am_settlement",
        "ForwardPrice": "F",
    })
    forwards = forwards[
        ["date", "exdate", "am_settlement", "F"]
    ].drop_duplicates(["date", "exdate", "am_settlement"])

    rates = rates_raw.copy()
    rates["date"] = pd.to_datetime(rates["date"])
    rates["days"] = pd.to_numeric(rates["days"], errors="coerce")
    rates["rate_decimal"] = pd.to_numeric(rates["rate"], errors="coerce") / 100.0
    rate_curves = {
        d: (g["days"].to_numpy(float), g["rate_decimal"].to_numpy(float))
        for d, g in rates.dropna().sort_values(["date", "days"]).groupby("date")
    }

    options = options.merge(spot, on="date", how="left").merge(
        forwards,
        on=["date", "exdate", "am_settlement"],
        how="left",
        validate="many_to_one",
    )
    options["r"] = [
        float(np.interp(n, *rate_curves[d])) if d in rate_curves else np.nan
        for d, n in zip(options["date"], options["T_days"])
    ]
    options["D"] = np.exp(-options["r"] * options["T"])
    options["carry"] = np.log(options["F"] / options["S"]) / options["T"]
    options["q"] = options["r"] - options["carry"]
    options["k"] = np.log(options["K"] / options["F"])

    options = options.dropna(subset=["S", "F", "r", "D", "carry", "k"])
    options = options.loc[options["k"].abs() <= MAX_ABS_LOG_MONEYNESS].copy()

    call_lb = options["D"] * np.maximum(options["F"] - options["K"], 0.0)
    call_ub = options["D"] * options["F"]
    put_lb = options["D"] * np.maximum(options["K"] - options["F"], 0.0)
    put_ub = options["D"] * options["K"]
    lb = np.where(options["cp_flag"].eq("C"), call_lb, put_lb)
    ub = np.where(options["cp_flag"].eq("C"), call_ub, put_ub)
    options = options.loc[
        (options["mid"] >= lb - 1e-6) & (options["mid"] <= ub + 1e-6)
    ].copy()

    return options.sort_values(
        ["date", "exdate", "am_settlement", "K", "cp_flag"]
    ).reset_index(drop=True)


def build_otm_surface(quotes):
    idx = [
        "date", "exdate", "am_settlement", "K", "T", "S", "F",
        "r", "q", "carry", "D", "k"
    ]
    vals = [
        "mid", "bid", "ask", "spread", "relative_spread",
        "iv_market", "delta_market", "optionid"
    ]

    wide = quotes.pivot_table(
        index=idx, columns="cp_flag", values=vals, aggfunc="first"
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    strict_call = wide["K"] >= wide["F"]
    near = wide["k"].abs() <= ATM_TIGHTER_SIDE_BAND
    have_c = wide.get("iv_market_C", pd.Series(np.nan, index=wide.index)).notna()
    have_p = wide.get("iv_market_P", pd.Series(np.nan, index=wide.index)).notna()
    c_spread = wide.get("spread_C", pd.Series(np.inf, index=wide.index))
    p_spread = wide.get("spread_P", pd.Series(np.inf, index=wide.index))

    choose_call = strict_call.copy()
    both_near = near & have_c & have_p
    choose_call = np.where(both_near, c_spread <= p_spread, choose_call)
    wide["cp_otm"] = np.where(choose_call, "C", "P")

    for base in vals:
        c = wide.get(f"{base}_C", pd.Series(np.nan, index=wide.index))
        p = wide.get(f"{base}_P", pd.Series(np.nan, index=wide.index))
        wide[f"{base}_otm"] = np.where(choose_call, c, p)

    wide = wide.dropna(subset=["iv_market_otm", "mid_otm", "optionid_otm"]).copy()

    sizes = wide.groupby(["date", "exdate", "am_settlement"]).size()
    valid = sizes[sizes >= MIN_POINTS_PER_SLICE].index

    return (
        wide.set_index(["date", "exdate", "am_settlement"])
        .loc[valid]
        .reset_index()
        .sort_values(["date", "T", "k"])
        .reset_index(drop=True)
    )


quotes_all = prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw)
otm = build_otm_surface(quotes_all)

print("Complete cleaned quotes:", len(quotes_all))
print("OTM calibration observations:", len(otm))
print("Trading dates:", otm["date"].nunique())
print("Expiry/settlement slices:", otm.groupby(["date", "exdate", "am_settlement"]).ngroups)


options : one_month_spx_option_price.csv
forwards: one_month_spx_forward_price.csv
spot    : one_month_spx_security_price.csv
rates   : one_month_spx_zero_coupon_yield_curve.csv
Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809


## 3. Curves, selected surface and deterministic daily 80/20 split


In [3]:
@dataclass
class DailyMarketCurve:
    date: pd.Timestamp
    am_settlement: int
    spot: float
    T_nodes: np.ndarray
    logD_nodes: np.ndarray
    logF_nodes: np.ndarray

    def __post_init__(self):
        self._logD = PchipInterpolator(self.T_nodes, self.logD_nodes, extrapolate=True)
        self._logF = PchipInterpolator(self.T_nodes, self.logF_nodes, extrapolate=True)

    def log_discount(self, t):
        return self._logD(np.maximum(np.asarray(t, float), 0.0))

    def log_forward_ratio(self, t):
        return self._logF(np.maximum(np.asarray(t, float), 0.0))

    def discount(self, t0, t1):
        return np.exp(self.log_discount(t1) - self.log_discount(t0))

    def forward(self, t):
        return self.spot * np.exp(self.log_forward_ratio(t))


class CurveBook:
    def __init__(self, spot_raw, forwards_raw, rates_raw):
        spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
        spot["date"] = pd.to_datetime(spot["date"])
        self.spot = spot.set_index("date")["close"].to_dict()

        self.forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
        self.forwards["date"] = pd.to_datetime(self.forwards["date"])
        self.forwards["expiration"] = pd.to_datetime(self.forwards["expiration"])

        self.rates = rates_raw.copy()
        self.rates["date"] = pd.to_datetime(self.rates["date"])
        self.rates["rate_decimal"] = pd.to_numeric(self.rates["rate"], errors="coerce") / 100.0
        self._cache = {}

    def get(self, date, am_settlement):
        date, am = pd.Timestamp(date), int(am_settlement)
        key = (date, am)
        if key in self._cache:
            return self._cache[key]

        S0 = float(self.spot[date])
        rg = self.rates.loc[self.rates["date"] == date, ["days", "rate_decimal"]].dropna().sort_values("days")
        T_r = rg["days"].to_numpy(float) / 365.0
        logD = -rg["rate_decimal"].to_numpy(float) * T_r

        fg = self.forwards.loc[
            (self.forwards["date"] == date) & (self.forwards["AMSettlement"] == am),
            ["expiration", "ForwardPrice"],
        ].copy()
        if len(fg) < 2:
            fg = self.forwards.loc[self.forwards["date"] == date, ["expiration", "ForwardPrice"]].copy()

        fg["T"] = (fg["expiration"] - date).dt.days / 365.0
        fg = (
            fg.loc[(fg["T"] >= 0) & (fg["ForwardPrice"] > 0)]
            .groupby("T", as_index=False)["ForwardPrice"].mean().sort_values("T")
        )
        T_f = fg["T"].to_numpy(float)
        logF = np.log(fg["ForwardPrice"].to_numpy(float) / S0)
        positive = T_f > 0
        T_f, logF = T_f[positive], logF[positive]

        T_all = np.unique(np.concatenate([[0.0], T_r, T_f]))
        logD_all = np.interp(T_all, np.r_[0.0, T_r], np.r_[0.0, logD])
        logF_all = np.interp(T_all, np.r_[0.0, T_f], np.r_[0.0, logF])

        curve = DailyMarketCurve(date, am, S0, T_all, logD_all, logF_all)
        self._cache[key] = curve
        return curve


curve_book = CurveBook(spot_raw, forwards_raw, rates_raw)


def sample_calibration_day(day, max_per_slice=MAX_QUOTES_PER_SLICE):
    pieces = []
    for _, g in day.groupby(["exdate", "am_settlement"]):
        g = g.sort_values("k")
        if len(g) <= max_per_slice:
            pieces.append(g)
        else:
            pos = np.unique(np.round(np.linspace(0, len(g)-1, max_per_slice)).astype(int))
            pieces.append(g.iloc[pos])
    return pd.concat(pieces, ignore_index=True)


def prepare_selected_universe(date):
    day = otm.loc[otm["date"] == pd.Timestamp(date)].copy()
    selected = sample_calibration_day(day).rename(columns={
        "mid_otm": "target_mid",
        "cp_otm": "target_cp",
        "optionid_otm": "optionid",
    }).reset_index(drop=True)
    selected["full_selected_id"] = np.arange(len(selected), dtype=int)
    return selected


def stratified_holdout(selected, fraction=HELDOUT_FRACTION, seed=SEED+77):
    """~80/20 within each expiry/settlement slice, spread across moneyness."""
    selected = selected.copy().reset_index(drop=True)
    is_holdout = np.zeros(len(selected), dtype=bool)
    rng = np.random.default_rng(seed)
    stride = max(2, int(round(1.0 / fraction)))

    for _, g in selected.groupby(["exdate", "am_settlement"], sort=False):
        ordered = g.sort_values("k").index.to_numpy()
        chosen = ordered[int(rng.integers(0, stride))::stride]
        if len(chosen) == 0:
            chosen = ordered[[len(ordered)//2]]
        if len(chosen) >= len(ordered):
            chosen = chosen[:-1]
        is_holdout[chosen] = True

    selected["split"] = np.where(is_holdout, "holdout", "train")
    train = selected.loc[~is_holdout].copy().reset_index(drop=True)
    holdout = selected.loc[is_holdout].copy().reset_index(drop=True)
    train["contract_id"] = np.arange(len(train), dtype=int)
    return selected, train, holdout


def daily_split(date):
    selected = prepare_selected_universe(date)
    return stratified_holdout(selected)


trading_dates = sorted(pd.to_datetime(quotes_all["date"].unique()))
if ANCHOR_DATE not in trading_dates:
    raise ValueError(f"Anchor date {ANCHOR_DATE.date()} is not in the option dataset.")

anchor_selected, anchor_train, anchor_holdout = daily_split(ANCHOR_DATE)
print("Anchor selected/train/validation:", len(anchor_selected), len(anchor_train), len(anchor_holdout))
print("Anchor maturities train/validation:", anchor_train["T"].nunique(), anchor_holdout["T"].nunique())


Anchor selected/train/validation: 997 798 199
Anchor maturities train/validation: 36 36


## 4. Locked maturity-head Neural LV and warm-start rules

The trunk is shared across all maturities. Each time interval has a small `50 -> 1 -> Softplus` volatility head.

For rolling recalibration from date \(d\) to \(d'\):

- copy the trunk exactly;
- a new head ending at \(T_{\text{new}}\) inherits the old head nearest to
  \(T_{\text{new}} + (d'-d)/365\), approximating the same calendar expiry;
- the control-variate hedge bank is new each day because the option identifiers/output dimension change.

March 1 is loaded from the completed v6 **raw-loss maturity-head** checkpoint. It is not retrained in the main v7 run.


In [4]:
class FeedForwardTrunk(nn.Module):
    def __init__(self, input_dim=2, hidden_layers=SDE_HIDDEN_LAYERS, width=SDE_WIDTH):
        super().__init__()
        layers, d = [], input_dim
        for _ in range(hidden_layers):
            layers += [nn.Linear(d, width), nn.ReLU()]
            d = width
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def sabate_weight_initialisation(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight.data, gain=1.5)


def _tx(t, x):
    if x.ndim == 1:
        x = x[:, None]
    if not torch.is_tensor(t):
        t = torch.full_like(x, float(t))
    elif t.ndim == 0:
        t = torch.full_like(x, float(t.item()))
    elif t.ndim == 1:
        t = t[:, None]
    return torch.cat([t, x], dim=1)


class MaturityHeadNeuralLocalVol(nn.Module):
    def __init__(self, maturity_boundaries):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.trunk = FeedForwardTrunk()
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(SDE_WIDTH, 1), nn.Softplus())
            for _ in self.boundaries
        ])
        self.apply(sabate_weight_initialisation)

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.heads)-1)

    def forward(self, t, x):
        return self.heads[self.interval_index(t)](self.trunk(_tx(t, x)))


class PerOptionHedgeNet(nn.Module):
    def __init__(self, n_options):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH)
        self.output = nn.Linear(HEDGE_WIDTH, int(n_options))
        self.apply(sabate_weight_initialisation)

    def forward_ids(self, t, x, option_ids):
        h = self.trunk(_tx(t, x))
        ids = torch.as_tensor(option_ids, dtype=torch.long, device=h.device)
        return h @ self.output.weight[ids].T + self.output.bias[ids][None, :]


class IntervalHedgeBank(nn.Module):
    def __init__(self, maturity_boundaries, n_options):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.nets = nn.ModuleList([
            PerOptionHedgeNet(n_options) for _ in self.boundaries
        ])

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.nets)-1)

    def forward_ids(self, t, x, option_ids):
        return self.nets[self.interval_index(t)].forward_ids(t, x, option_ids)


def set_requires_grad(module, value):
    for p in module.parameters():
        p.requires_grad_(value)


def safe_torch_load(path, map_location=DEVICE):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def transfer_rolling_model(previous, current, previous_date, current_date):
    """Copy trunk; shift heads by elapsed calendar time."""
    current.trunk.load_state_dict(previous.trunk.state_dict())
    elapsed = (pd.Timestamp(current_date) - pd.Timestamp(previous_date)).days / 365.0

    matched = []
    with torch.no_grad():
        for i, T_new in enumerate(current.boundaries):
            target_old_T = float(T_new) + elapsed
            j = int(np.argmin(np.abs(previous.boundaries - target_old_T)))
            current.heads[i].load_state_dict(previous.heads[j].state_dict())
            matched.append(abs(float(previous.boundaries[j]) - target_old_T))

    print(
        f"Warm-start {pd.Timestamp(previous_date).date()} -> {pd.Timestamp(current_date).date()}: "
        f"{len(current.heads)} heads, median maturity mismatch={np.median(matched):.5f}y"
    )


def locate_v6_anchor(date=ANCHOR_DATE):
    stamp = f"{pd.Timestamp(date):%Y%m%d}"
    exact = V6_CACHE_DIR / f"real_{stamp}_split80_20_ablation_rawloss_maturity_heads_resume.pt"
    if exact.exists():
        return exact
    matches = sorted(V6_CACHE_DIR.glob(f"*{stamp}*ablation_rawloss*maturity_heads*resume.pt"))
    return matches[0] if matches else None


def locate_v5_checkpoint(date=ANCHOR_DATE):
    stamp = f"{pd.Timestamp(date):%Y%m%d}"
    exact = V5_CACHE_DIR / f"stage1_shared_{stamp}_resume.pt"
    if exact.exists():
        return exact
    matches = sorted(V5_CACHE_DIR.glob(f"*{stamp}*resume*.pt")) if V5_CACHE_DIR.exists() else []
    return matches[0] if matches else None


def warm_start_from_v5(model, date=ANCHOR_DATE):
    """Only for optional CPU seed robustness; the main v7 run loads the v6 anchor."""
    path = locate_v5_checkpoint(date)
    if path is None:
        raise FileNotFoundError("No v5 checkpoint found for warm-start seed robustness.")

    payload = safe_torch_load(path, map_location="cpu")
    state = payload.get("final_model_state") or payload.get("model_state")
    trunk_state = {k[len("trunk."):]: v for k, v in state.items() if k.startswith("trunk.")}
    model.trunk.load_state_dict(trunk_state, strict=True)

    w, b = state.get("out.0.weight"), state.get("out.0.bias")
    if w is not None and b is not None:
        with torch.no_grad():
            for head in model.heads:
                head[0].weight.copy_(w)
                head[0].bias.copy_(b)
    print("Warm-started optional seed run from:", path)


## 5. Calibration context, raw-price objective and Sabaté control variates


In [5]:
def build_time_grid(maturities, steps_per_year=BASE_STEPS_PER_YEAR):
    maturities = np.sort(np.unique(np.asarray(maturities, float)))
    max_T = float(maturities[-1])
    base = np.arange(0.0, max_T + 0.5/steps_per_year, 1.0/steps_per_year)
    grid = np.unique(np.r_[0.0, base[base <= max_T + 1e-12], maturities, max_T])
    grid.sort()
    return grid


def nearest_grid_indices(grid, times):
    grid, times = np.asarray(grid), np.asarray(times)
    idx = np.clip(np.searchsorted(grid, times), 0, len(grid)-1)
    left = np.maximum(idx-1, 0)
    idx = np.where(np.abs(grid[left]-times) < np.abs(grid[idx]-times), left, idx)
    if np.max(np.abs(grid[idx]-times)) > 1e-10:
        raise RuntimeError("A listed maturity is not represented exactly on the simulation grid.")
    return idx


@dataclass
class CalibrationContext:
    date: pd.Timestamp
    spot: float
    contracts: pd.DataFrame
    maturity_boundaries: np.ndarray
    grid_np: np.ndarray
    grid: torch.Tensor
    maturity_index: torch.Tensor
    stage_index: np.ndarray
    stage_tensor: torch.Tensor
    K: torch.Tensor
    F_T: torch.Tensor
    D_T: torch.Tensor
    target: torch.Tensor
    cp_sign: torch.Tensor
    am: torch.Tensor
    F_grid: torch.Tensor
    D_grid: torch.Tensor


def build_calibration_context(date, contracts):
    date = pd.Timestamp(date)
    contracts = contracts.copy().reset_index(drop=True)
    contracts["contract_id"] = np.arange(len(contracts), dtype=int)

    boundaries = np.sort(contracts["T"].unique())
    grid_np = build_time_grid(boundaries)
    maturity_idx = nearest_grid_indices(grid_np, contracts["T"].to_numpy(float))
    stage_map = {round(float(T), 12): i for i, T in enumerate(boundaries)}
    stage_index = np.array([stage_map[round(float(T), 12)] for T in contracts["T"]], dtype=int)

    F_grid = np.empty((2, len(grid_np)))
    D_grid = np.empty((2, len(grid_np)))
    for am in [0, 1]:
        curve = curve_book.get(date, am)
        F_grid[am] = curve.forward(grid_np)
        D_grid[am] = curve.discount(0.0, grid_np)

    return CalibrationContext(
        date=date,
        spot=float(contracts["S"].iloc[0]),
        contracts=contracts,
        maturity_boundaries=boundaries,
        grid_np=grid_np,
        grid=torch.tensor(grid_np, dtype=torch.float32, device=DEVICE),
        maturity_index=torch.tensor(maturity_idx, dtype=torch.long, device=DEVICE),
        stage_index=stage_index,
        stage_tensor=torch.tensor(stage_index, dtype=torch.long, device=DEVICE),
        K=torch.tensor(contracts["K"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        F_T=torch.tensor(contracts["F"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        D_T=torch.tensor(contracts["D"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        target=torch.tensor(contracts["target_mid"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        cp_sign=torch.tensor(np.where(contracts["target_cp"].eq("C"), 1.0, -1.0), dtype=torch.float32, device=DEVICE),
        am=torch.tensor(contracts["am_settlement"].to_numpy(int), dtype=torch.long, device=DEVICE),
        F_grid=torch.tensor(F_grid, dtype=torch.float32, device=DEVICE),
        D_grid=torch.tensor(D_grid, dtype=torch.float32, device=DEVICE),
    )


def antithetic_normals(total_paths, n_steps, seed):
    half = (total_paths + 1) // 2
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed))
    z = torch.randn(half, n_steps, generator=gen, dtype=torch.float32)
    return torch.cat([z, -z], dim=0)[:total_paths].to(DEVICE)


def empirical_payoff_mean(samples, strike_norm, cp_flag):
    x = np.sort(np.asarray(samples, float))
    n = len(x)
    prefix = np.r_[0.0, np.cumsum(x)]
    k = np.asarray(strike_norm, float)
    idx = np.searchsorted(x, k, side="right")
    below, above = prefix[idx], prefix[-1] - prefix[idx]
    call = (above - k*(n-idx)) / n
    put = (k*idx - below) / n
    return np.where(np.asarray(cp_flag) == "C", call, put)


def maturity_balanced_mean(values, stages):
    return torch.stack([values[stages == s].mean() for s in torch.unique(stages)]).mean()


class MaturityBalancedSampler:
    def __init__(self, ctx, per_maturity, seed):
        self.groups = [np.where(ctx.stage_index == j)[0] for j in range(len(ctx.maturity_boundaries))]
        self.per_maturity = per_maturity
        self.rng = np.random.default_rng(seed)

    def draw(self):
        out = []
        for g in self.groups:
            n = min(self.per_maturity, len(g))
            out.extend(self.rng.choice(g, size=n, replace=False))
        return np.asarray(out, dtype=int)


def simulate_estimators(model, hedge, ctx, contract_ids, z, phase):
    ids_np = np.asarray(contract_ids, dtype=int)
    ids = torch.as_tensor(ids_np, dtype=torch.long, device=DEVICE)
    maturity_idx = ctx.maturity_index[ids]
    K, F_T, D_T = ctx.K[ids], ctx.F_T[ids], ctx.D_T[ids]
    cp, am = ctx.cp_sign[ids], ctx.am[ids]

    B, M = z.shape[0], len(ids_np)
    x = torch.ones((B, 1), dtype=torch.float32, device=DEVICE)
    cv = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)
    raw = torch.zeros((B, M), dtype=torch.float32, device=DEVICE)

    for step in range(int(maturity_idx.max().item())):
        t = ctx.grid[step]
        dt = ctx.grid[step+1] - ctx.grid[step]
        sqrt_dt = torch.sqrt(dt)
        sigma = model(t, x) if phase == "sde" else model(t, x).detach()
        dW = sqrt_dt * z[:, step:step+1]

        active = torch.where(maturity_idx > step)[0]
        if len(active):
            active_ids = ids_np[active.detach().cpu().numpy()]
            h = hedge.forward_ids(t, x.detach(), active_ids)
            if phase == "sde":
                h = h.detach()
            stochastic_piece = x.detach() * sigma.detach() * dW
            discounted_forward = ctx.D_grid[am[active], step] * ctx.F_grid[am[active], step]
            cv[:, active] += h * stochastic_piece * discounted_forward[None, :]

        a = x * sigma
        x = x + a*dW / (1.0 + torch.abs(x.detach()*sigma.detach())*sqrt_dt)

        matured = torch.where(maturity_idx == step+1)[0]
        if len(matured):
            S_T = x * F_T[matured][None, :]
            strike = K[matured][None, :]
            call = torch.clamp(S_T-strike, min=0.0)
            put = torch.clamp(strike-S_T, min=0.0)
            raw[:, matured] = torch.where((cp[matured] > 0)[None, :], call, put) * D_T[matured][None, :]

    return raw, raw-cv


@torch.no_grad()
def simulate_terminal_x(model, ctx, z):
    x = torch.ones((z.shape[0], 1), dtype=torch.float32, device=DEVICE)
    stage_by_grid = {
        int(nearest_grid_indices(ctx.grid_np, [T])[0]): j
        for j, T in enumerate(ctx.maturity_boundaries)
    }
    terminal = {}
    for step in range(len(ctx.grid_np)-1):
        t = ctx.grid[step]
        dt = ctx.grid[step+1] - ctx.grid[step]
        sigma = model(t, x)
        dW = torch.sqrt(dt) * z[:, step:step+1]
        a = x * sigma
        x = x + a*dW / (1.0 + torch.abs(a)*torch.sqrt(dt))
        if step+1 in stage_by_grid:
            terminal[stage_by_grid[step+1]] = x.squeeze(1).detach().cpu().numpy().copy()
    return terminal


def selected_prices_from_terminal(ctx, terminal):
    rows = []
    for stage in range(len(ctx.maturity_boundaries)):
        ids = np.where(ctx.stage_index == stage)[0]
        g = ctx.contracts.iloc[ids]
        payoff = empirical_payoff_mean(
            terminal[stage],
            g["K"].to_numpy(float) / g["F"].to_numpy(float),
            g["target_cp"].to_numpy(),
        )
        price = g["D"].to_numpy(float) * g["F"].to_numpy(float) * payoff
        tmp = g[["contract_id", "optionid", "T", "k", "target_mid"]].copy()
        tmp["model_price"] = price
        tmp["raw_error"] = price - tmp["target_mid"].to_numpy(float)
        rows.append(tmp)
    return pd.concat(rows, ignore_index=True)


def objective_diagnostics(model, ctx, z_fixed):
    table = selected_prices_from_terminal(ctx, simulate_terminal_x(model, ctx, z_fixed))
    e = table["raw_error"].to_numpy(float)
    table["objective_error"] = e / ctx.spot
    by_T = table.groupby("T")["objective_error"].apply(
        lambda x: float(np.mean(np.asarray(x, float)**2))
    )
    return {
        "objective_rmse": float(np.sqrt(by_T.mean())),
        "raw_rmse": float(np.sqrt(np.mean(e**2))),
        "raw_mae": float(np.mean(np.abs(e))),
        "table": table,
    }


def choose_even_ids(n_total, n):
    if n is None or n >= n_total:
        return np.arange(n_total, dtype=int)
    return np.unique(np.round(np.linspace(0, n_total-1, n)).astype(int))


@torch.no_grad()
def cv_variance_ratio(model, hedge, ctx, n_contracts=64, paths=8_192, seed=SEED+9000):
    ids = choose_even_ids(len(ctx.contracts), n_contracts)
    z = antithetic_normals(paths, len(ctx.grid_np)-1, seed)
    raw, cv = simulate_estimators(model, hedge, ctx, ids, z, phase="sde")
    ratio = torch.median(
        cv.var(0, unbiased=False) / torch.clamp(raw.var(0, unbiased=False), min=1e-12)
    )
    out = float(ratio.cpu())
    del z, raw, cv
    gc.collect()
    return out


anchor_ctx = build_calibration_context(ANCHOR_DATE, anchor_train)
print("Anchor training contracts:", len(anchor_ctx.contracts))
print("Anchor maturity heads:", len(anchor_ctx.maturity_boundaries))
print("Anchor simulation steps:", len(anchor_ctx.grid_np)-1)


Anchor training contracts: 798
Anchor maturity heads: 36
Anchor simulation steps: 129


## 6. Rolling trainer — resumable, independently reranked, training-only early stopping

A nominal epoch alternates between Neural-SDE and hedge/control-variate optimisation, as in v6. Therefore 120 nominal epochs contain about 60 SDE parameter-update epochs and 60 hedge-network epochs.

Early stopping is checked only after fixed-path evaluations on the **current day's training contracts**. A meaningful relative improvement resets patience; the daily 20% hold-out is never inspected by this trainer.


In [6]:
@dataclass(frozen=True)
class TrainConfig:
    epochs: int = ROLLING_EPOCHS
    paths_per_step: int = PATHS_PER_STEP
    steps_per_epoch: int = STEPS_PER_EPOCH
    quotes_per_maturity: int = QUOTES_PER_MATURITY_PER_STEP
    monitor_paths: int = MONITOR_PATHS
    eval_every_sde: int = EVAL_EVERY_SDE
    top_k: int = TOP_K_CHECKPOINTS
    rerank_paths: int = RERANK_PATHS
    save_every: int = SAVE_EVERY_EPOCHS
    sde_lr: float = SDE_LR
    hedge_lr: float = HEDGE_LR
    early_min_epochs: int = EARLY_MIN_EPOCHS
    early_patience: int = EARLY_PATIENCE
    early_rel_delta: float = EARLY_REL_DELTA
    seed_offset: int = 0


def checkpoint_file(run_tag):
    return CACHE_DIR / f"{run_tag}_resume.pt"


def candidate_folder(run_tag):
    folder = CACHE_DIR / f"candidates_{run_tag}"
    folder.mkdir(exist_ok=True)
    return folder


def save_training_state(path, next_epoch, model, hedge, opt_sde, opt_hedge, history,
                        candidates, best_monitor, bad_checks, training_finished=False,
                        completed=False, stop_reason=None, final_model_state=None,
                        final_hedge_state=None, rerank_records=None):
    torch.save({
        "version": "v7",
        "next_epoch": int(next_epoch),
        "training_finished": bool(training_finished),
        "completed": bool(completed),
        "stop_reason": stop_reason,
        "model_state": model.state_dict(),
        "hedge_state": hedge.state_dict(),
        "opt_sde": opt_sde.state_dict(),
        "opt_hedge": opt_hedge.state_dict(),
        "history": history,
        "candidates": candidates,
        "best_monitor": float(best_monitor),
        "bad_checks": int(bad_checks),
        "boundaries": np.asarray(model.boundaries, float),
        "final_model_state": final_model_state,
        "final_hedge_state": final_hedge_state,
        "rerank_records": rerank_records,
    }, path)


def maybe_store_candidate(model, hedge, candidates, epoch, diag, run_tag, top_k):
    score = float(diag["objective_rmse"])
    if len(candidates) >= top_k and score >= max(x["monitor_objective_rmse"] for x in candidates):
        return candidates

    folder = candidate_folder(run_tag)
    cpath = folder / f"candidate_epoch_{int(epoch):04d}.pt"
    torch.save({
        "epoch": int(epoch),
        "monitor_objective_rmse": score,
        "monitor_raw_rmse": float(diag["raw_rmse"]),
        "model_state": model.state_dict(),
        "hedge_state": hedge.state_dict(),
    }, cpath)

    candidates = list(candidates) + [{
        "epoch": int(epoch),
        "monitor_objective_rmse": score,
        "monitor_raw_rmse": float(diag["raw_rmse"]),
        "path": str(cpath),
    }]
    candidates = sorted(candidates, key=lambda x: x["monitor_objective_rmse"])
    while len(candidates) > top_k:
        removed = candidates.pop(-1)
        try:
            Path(removed["path"]).unlink(missing_ok=True)
        except OSError:
            pass
    return candidates


@torch.no_grad()
def rerank_candidates(ctx, candidates, cfg):
    if not candidates:
        raise RuntimeError("No monitor-selected checkpoints available to rerank.")

    z = antithetic_normals(
        cfg.rerank_paths, len(ctx.grid_np)-1, SEED + 5000 + cfg.seed_offset
    )
    temp = MaturityHeadNeuralLocalVol(ctx.maturity_boundaries).to(DEVICE).eval()
    rows = []

    for c in tqdm(candidates, desc="Independent checkpoint rerank", unit="model"):
        p = safe_torch_load(c["path"])
        temp.load_state_dict(p["model_state"])
        d = objective_diagnostics(temp, ctx, z)
        rows.append({
            **c,
            "rerank_objective_rmse": d["objective_rmse"],
            "rerank_raw_rmse": d["raw_rmse"],
        })

    del z, temp
    gc.collect()

    table = pd.DataFrame(rows).sort_values(
        ["rerank_objective_rmse", "monitor_objective_rmse"]
    ).reset_index(drop=True)
    table["independent_rank"] = np.arange(1, len(table)+1)
    return table, table.iloc[0].to_dict()


def train_rolling_day(ctx, cfg, run_tag, previous_model=None, previous_date=None,
                      warm_start_v5_date=None):
    """Train only on ctx.contracts. Daily hold-out data never enters this function."""
    path = checkpoint_file(run_tag)
    set_seed(SEED + cfg.seed_offset)

    model = MaturityHeadNeuralLocalVol(ctx.maturity_boundaries).to(DEVICE)
    hedge = IntervalHedgeBank(ctx.maturity_boundaries, len(ctx.contracts)).to(DEVICE)
    opt_sde = torch.optim.Adam(model.parameters(), lr=cfg.sde_lr)
    opt_hedge = torch.optim.Adam(hedge.parameters(), lr=cfg.hedge_lr)

    start_epoch, history, candidates = 0, [], []
    best_monitor, bad_checks = np.inf, 0
    training_finished = False
    stop_reason = None

    if path.exists() and not FORCE_RETRAIN:
        p = safe_torch_load(path)
        if p.get("completed", False):
            model.load_state_dict(p["final_model_state"])
            hedge.load_state_dict(p["final_hedge_state"])
            model.eval(); hedge.eval()
            set_requires_grad(model, False); set_requires_grad(hedge, False)
            print("Loaded completed v7 checkpoint:", path)
            return model, hedge, pd.DataFrame(p["history"]), pd.DataFrame(p["rerank_records"]), p.get("stop_reason")

        model.load_state_dict(p["model_state"])
        hedge.load_state_dict(p["hedge_state"])
        opt_sde.load_state_dict(p["opt_sde"])
        opt_hedge.load_state_dict(p["opt_hedge"])
        start_epoch = int(p["next_epoch"])
        history = list(p["history"])
        candidates = list(p.get("candidates", []))
        best_monitor = float(p.get("best_monitor", np.inf))
        bad_checks = int(p.get("bad_checks", 0))
        training_finished = bool(p.get("training_finished", False))
        stop_reason = p.get("stop_reason")
        print("Resuming v7", run_tag, "at epoch", start_epoch+1 if not training_finished else "reranking")
    else:
        if FORCE_RETRAIN:
            for f in candidate_folder(run_tag).glob("candidate_epoch_*.pt"):
                f.unlink(missing_ok=True)
        if previous_model is not None:
            if previous_date is None:
                raise ValueError("previous_date is required with previous_model.")
            transfer_rolling_model(previous_model, model, previous_date, ctx.date)
        elif warm_start_v5_date is not None:
            warm_start_from_v5(model, warm_start_v5_date)

    sampler = MaturityBalancedSampler(ctx, cfg.quotes_per_maturity, SEED + 10 + cfg.seed_offset)
    z_monitor = antithetic_normals(
        cfg.monitor_paths, len(ctx.grid_np)-1, SEED + 100 + cfg.seed_offset
    )

    if not training_finished:
        sde_count = sum(1 for e in range(start_epoch) if e % 2 == 0)
        bar = tqdm(
            range(start_epoch, cfg.epochs),
            desc=f"v7 {run_tag}",
            unit="epoch",
            dynamic_ncols=True,
            mininterval=1.0,
        )

        for epoch in bar:
            phase = "sde" if epoch % 2 == 0 else "hedge"
            set_requires_grad(model, phase == "sde")
            set_requires_grad(hedge, phase == "hedge")
            losses = []

            for _ in range(cfg.steps_per_epoch):
                ids = sampler.draw()
                z = torch.randn(
                    cfg.paths_per_step, len(ctx.grid_np)-1,
                    dtype=torch.float32, device=DEVICE
                )
                opt_sde.zero_grad(set_to_none=True)
                opt_hedge.zero_grad(set_to_none=True)
                raw, cv = simulate_estimators(model, hedge, ctx, ids, z, phase)

                ids_t = torch.as_tensor(ids, dtype=torch.long, device=DEVICE)
                stages = ctx.stage_tensor[ids_t]

                if phase == "sde":
                    # Spot is a date-wide constant: same minimiser as raw-price MSE.
                    residual = (cv.mean(0) - ctx.target[ids_t]) / ctx.spot
                    loss = maturity_balanced_mean(residual.square(), stages)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    opt_sde.step()
                else:
                    variance = torch.var(cv / ctx.spot, dim=0, unbiased=False)
                    loss = maturity_balanced_mean(variance, stages)
                    loss.backward()
                    nn.utils.clip_grad_norm_(hedge.parameters(), 3.0)
                    opt_hedge.step()

                losses.append(float(loss.detach().cpu()))
                del raw, cv, z

            diag = None
            should_stop = False

            if phase == "sde":
                sde_count += 1
                if sde_count == 1 or sde_count % cfg.eval_every_sde == 0:
                    set_requires_grad(model, False)
                    set_requires_grad(hedge, False)
                    diag = objective_diagnostics(model, ctx, z_monitor)
                    candidates = maybe_store_candidate(
                        model, hedge, candidates, epoch+1, diag, run_tag, cfg.top_k
                    )

                    score = float(diag["objective_rmse"])
                    meaningful = score < best_monitor * (1.0 - cfg.early_rel_delta)
                    if meaningful:
                        best_monitor, bad_checks = score, 0
                    else:
                        bad_checks += 1

                    if (epoch+1) >= cfg.early_min_epochs and bad_checks >= cfg.early_patience:
                        should_stop = True
                        stop_reason = (
                            f"early_stop: no >= {cfg.early_rel_delta:.2%} monitor improvement "
                            f"for {cfg.early_patience} checks"
                        )

            row = {
                "epoch": epoch+1,
                "phase": phase,
                "train_loss": float(np.mean(losses)),
                "sde_lr": cfg.sde_lr,
                "hedge_lr": cfg.hedge_lr,
                "objective_rmse": np.nan if diag is None else diag["objective_rmse"],
                "raw_rmse": np.nan if diag is None else diag["raw_rmse"],
                "bad_checks": bad_checks,
            }
            history.append(row)

            post = {"phase": phase, "loss": f"{row['train_loss']:.2e}"}
            if diag is not None:
                post["RMSE"] = f"{diag['raw_rmse']:.3f}"
                post["patience"] = f"{bad_checks}/{cfg.early_patience}"
            bar.set_postfix(post)

            if (epoch+1) % cfg.save_every == 0 or should_stop or epoch == cfg.epochs-1:
                save_training_state(
                    path, epoch+1, model, hedge, opt_sde, opt_hedge,
                    history, candidates, best_monitor, bad_checks,
                    training_finished=(should_stop or epoch == cfg.epochs-1),
                    completed=False, stop_reason=stop_reason,
                )

            if should_stop:
                bar.close()
                print("Early stopping:", stop_reason, "| epoch", epoch+1)
                training_finished = True
                break

            gc.collect()

        if not training_finished:
            training_finished = True
            stop_reason = stop_reason or "max_epochs"

        final_epoch = int(history[-1]["epoch"])
        save_training_state(
            path, final_epoch, model, hedge, opt_sde, opt_hedge,
            history, candidates, best_monitor, bad_checks,
            training_finished=True, completed=False, stop_reason=stop_reason or "max_epochs",
        )

    del z_monitor
    gc.collect()

    rerank_table, best = rerank_candidates(ctx, candidates, cfg)
    print(
        f"Selected epoch {int(best['epoch'])}: "
        f"rerank raw RMSE={best['rerank_raw_rmse']:.4f}"
    )

    p = safe_torch_load(best["path"])
    model.load_state_dict(p["model_state"])
    hedge.load_state_dict(p["hedge_state"])
    model.eval(); hedge.eval()
    set_requires_grad(model, False); set_requires_grad(hedge, False)

    final_model_state = copy.deepcopy(model.state_dict())
    final_hedge_state = copy.deepcopy(hedge.state_dict())
    final_epoch = int(history[-1]["epoch"])

    save_training_state(
        path, final_epoch, model, hedge, opt_sde, opt_hedge,
        history, candidates, best_monitor, bad_checks,
        training_finished=True, completed=True,
        stop_reason=stop_reason or "max_epochs",
        final_model_state=final_model_state,
        final_hedge_state=final_hedge_state,
        rerank_records=rerank_table.to_dict("records"),
    )
    return model, hedge, pd.DataFrame(history), rerank_table, stop_reason or "max_epochs"


ROLLING_CFG = TrainConfig()


## 7. Pricing, implied-volatility, delta and future-date evaluation helpers


In [7]:
def tkey(t):
    return round(float(t), 12)


def bs_forward_price(F, K, D, T, sigma, cp="C"):
    F, K, D, T, sigma = map(float, (F, K, D, T, sigma))
    if T <= 0 or sigma <= 0:
        return D*max(F-K, 0.0) if cp == "C" else D*max(K-F, 0.0)
    srt = sigma * math.sqrt(T)
    d1 = (math.log(F/K) + 0.5*srt*srt) / srt
    d2 = d1 - srt
    call = D * (F*float(ndtr(d1)) - K*float(ndtr(d2)))
    return call if cp == "C" else call - D*(F-K)


def implied_vol_from_price(price, F, K, D, T, cp):
    intrinsic = D*max(F-K, 0.0) if cp == "C" else D*max(K-F, 0.0)
    upper = D*F if cp == "C" else D*K
    if not (intrinsic-1e-8 <= price <= upper+1e-8) or T <= 0:
        return np.nan
    try:
        return float(brentq(
            lambda s: bs_forward_price(F, K, D, T, s, cp) - price,
            1e-5, 5.0, maxiter=100
        ))
    except ValueError:
        return np.nan


@torch.no_grad()
def simulate_same_day_x(model, maturities, total_paths, chunk_paths, seed, progress=True):
    if total_paths % 2:
        raise ValueError("total_paths must be even for antithetic evaluation.")

    grid = build_time_grid(maturities)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)

    collected = {key: [] for key in maturity_idx}
    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)
    total_pairs, chunk_pairs, done = total_paths//2, max(1, chunk_paths//2), 0
    bar = tqdm(total=total_paths, desc="Same-day MC", unit="path", disable=not progress)

    while done < total_pairs:
        n_pairs = min(chunk_pairs, total_pairs-done)
        z_half = torch.randn(n_pairs, len(grid)-1, generator=gen, dtype=torch.float32)
        z = torch.cat([z_half, -z_half], 0).to(DEVICE)
        x = torch.ones((2*n_pairs, 1), dtype=torch.float32, device=DEVICE)

        for step in range(len(grid)-1):
            t = float(grid[step])
            dt = float(grid[step+1] - grid[step])
            sigma = model(torch.full_like(x, t), x)
            dW = math.sqrt(dt) * z[:, step:step+1]
            a = x * sigma
            x = x + a*dW / (1.0 + torch.abs(a)*math.sqrt(dt))

            if step+1 in index_to_keys:
                arr = x.squeeze(1).detach().cpu().numpy()
                paired = np.stack([arr[:n_pairs], arr[n_pairs:]], axis=1)
                for key in index_to_keys[step+1]:
                    collected[key].append(paired.copy())

        done += n_pairs
        bar.update(2*n_pairs)
        del z, x
        gc.collect()

    bar.close()
    return {k: np.concatenate(v, axis=0) for k, v in collected.items()}


def price_rows_same_day(rows, samples_by_T, base_col="mid"):
    out = rows.copy().reset_index(drop=True)
    price = np.empty(len(out))

    for T, idx in out.groupby("T").groups.items():
        pos = np.asarray(list(idx), dtype=int)
        g = out.loc[idx]
        samples = samples_by_T[tkey(T)].reshape(-1)
        payoff = empirical_payoff_mean(
            samples,
            g["K"].to_numpy(float) / g["F"].to_numpy(float),
            g["cp_flag"].to_numpy(),
        )
        price[pos] = g["D"].to_numpy(float) * g["F"].to_numpy(float) * payoff

    out["neural_lv_price"] = price
    out["pricing_error"] = price - out[base_col].to_numpy(float)
    out["inside_spread"] = out["neural_lv_price"].between(out["bid"], out["ask"])
    return out


def add_model_iv(rows, desc="Model IV"):
    out = rows.copy()
    iv = [
        implied_vol_from_price(r.neural_lv_price, r.F, r.K, r.D, r.T, r.cp_flag)
        for r in tqdm(out.itertuples(index=False), total=len(out), desc=desc, unit="opt")
    ]
    out["iv_model"] = iv
    market_col = "iv_market" if "iv_market" in out.columns else "iv_market_otm"
    out["iv_error"] = out["iv_model"] - out[market_col].to_numpy(float)
    return out


def delta_rows_from_terminal(rows, samples_by_T, bump_bps=DELTA_BUMP_BPS):
    out = rows.copy().reset_index(drop=True)

    for bps in bump_bps:
        eps = float(bps) * 1e-4
        delta = np.empty(len(out))

        for T, idx in out.groupby("T").groups.items():
            pos = np.asarray(list(idx), dtype=int)
            g = out.loc[idx]
            samples = samples_by_T[tkey(T)].reshape(-1)

            F = g["F"].to_numpy(float)
            K = g["K"].to_numpy(float)
            D = g["D"].to_numpy(float)
            cp = g["cp_flag"].to_numpy()
            S = g["S"].to_numpy(float)

            Fu, Fd = F*(1+eps), F*(1-eps)
            pu = D*Fu*empirical_payoff_mean(samples, K/Fu, cp)
            pdn = D*Fd*empirical_payoff_mean(samples, K/Fd, cp)
            delta[pos] = (pu-pdn) / (2*eps*S)

        out[f"delta_{bps}bp"] = delta

    headline = f"delta_{DELTA_HEADLINE_BPS}bp"
    out["neural_lv_delta"] = out[headline]
    others = [f"delta_{b}bp" for b in bump_bps if b != DELTA_HEADLINE_BPS]
    out["delta_bump_instability"] = np.max(
        np.column_stack([np.abs(out[c]-out[headline]) for c in others]), axis=1
    )
    return out


def price_summary(df, error_col="pricing_error", market_col="mid", inside_col="inside_spread"):
    e = df[error_col].to_numpy(float)
    market = np.maximum(df[market_col].to_numpy(float), 1e-12)

    out = {
        "n": len(df),
        "MSE": float(np.mean(e**2)),
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "MAE": float(np.mean(np.abs(e))),
        "Rel_MAE_pct": float(np.mean(np.abs(e)/market) * 100.0),
    }
    if inside_col in df:
        out["inside_bid_ask"] = float(df[inside_col].mean())
    if "iv_error" in df:
        ie = df["iv_error"].dropna().to_numpy(float)
        out["IV_RMSE"] = float(np.sqrt(np.mean(ie**2))) if len(ie) else np.nan
    return out


def selected_quote_ids(selected):
    return set(selected["optionid"].astype(int))


def evaluate_same_day(model, date, train, holdout, paths=FINAL_MC_PATHS, seed=SEED+7000):
    date = pd.Timestamp(date)
    full = quotes_all.loc[quotes_all["date"] == date].copy().reset_index(drop=True)
    otm_day = otm.loc[otm["date"] == date].rename(columns={
        "mid_otm": "mid", "bid_otm": "bid", "ask_otm": "ask",
        "cp_otm": "cp_flag", "optionid_otm": "optionid",
    }).copy().reset_index(drop=True)

    samples = simulate_same_day_x(
        model, full["T"].unique(), paths, FINAL_PATH_CHUNK, seed
    )
    full_eval = delta_rows_from_terminal(price_rows_same_day(full, samples), samples)
    otm_eval = add_model_iv(price_rows_same_day(otm_day, samples), f"{date.date()} full OTM IV")

    train_eval = add_model_iv(
        full_eval.loc[full_eval["optionid"].astype(int).isin(selected_quote_ids(train))].copy(),
        f"{date.date()} train IV",
    )
    holdout_eval = add_model_iv(
        full_eval.loc[full_eval["optionid"].astype(int).isin(selected_quote_ids(holdout))].copy(),
        f"{date.date()} holdout IV",
    )

    hierarchy = pd.DataFrame({
        "selected_train_80": price_summary(train_eval),
        "selected_holdout_20": price_summary(holdout_eval),
        "full_OTM_strip": price_summary(otm_eval),
        "complete_clean_chain": price_summary(full_eval),
    }).T

    del samples
    gc.collect()
    return hierarchy, full_eval, train_eval, holdout_eval, otm_eval


def build_evaluation_pairs(quotes):
    dates = sorted(quotes["date"].unique())
    next_map = {dates[i]: dates[i+1] for i in range(len(dates)-1)}
    left = quotes.copy()
    left["next_date"] = left["date"].map(next_map)

    cols = [
        "date", "optionid", "cp_flag", "K", "exdate", "am_settlement",
        "mid", "bid", "ask", "S", "F", "D", "r", "q", "carry", "T", "k"
    ]
    right = quotes[cols].rename(columns={
        "date": "next_date", "mid": "mid_next", "bid": "bid_next",
        "ask": "ask_next", "S": "S_next", "F": "F_next", "D": "D_next",
        "r": "r_next", "q": "q_next", "carry": "carry_next",
        "T": "T_next", "k": "k_next",
    })
    return left.merge(
        right,
        on=["next_date", "optionid", "cp_flag", "K", "exdate", "am_settlement"],
        how="inner",
        validate="many_to_one",
    )


@torch.no_grad()
def simulate_frozen_to_date(model, curve0, curve1, elapsed, maturities_remaining,
                            total_paths, chunk_paths, seed, progress=True):
    if total_paths % 2:
        raise ValueError("total_paths must be even.")

    grid = build_time_grid(maturities_remaining)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities_remaining, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)

    collected = {key: [] for key in maturity_idx}
    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)
    total_pairs, chunk_pairs, done = total_paths//2, max(1, chunk_paths//2), 0
    bar = tqdm(
        total=total_paths,
        desc=f"Future-date MC AM={curve1.am_settlement}",
        unit="path",
        disable=not progress,
    )

    while done < total_pairs:
        n_pairs = min(chunk_pairs, total_pairs-done)
        z_half = torch.randn(n_pairs, len(grid)-1, generator=gen, dtype=torch.float32)
        z = torch.cat([z_half, -z_half], 0).to(DEVICE)
        y = torch.ones((2*n_pairs, 1), dtype=torch.float32, device=DEVICE)

        for step in range(len(grid)-1):
            v = float(grid[step])
            dt = float(grid[step+1] - grid[step])
            abs_t = elapsed + v

            Fnew = float(curve1.forward(v))
            Fold = float(curve0.forward(abs_t))
            x_old = y * (Fnew/Fold)

            sigma = model(torch.full_like(y, abs_t), x_old)
            dW = math.sqrt(dt) * z[:, step:step+1]
            a = y * sigma
            y = y + a*dW / (1.0 + torch.abs(a)*math.sqrt(dt))

            if step+1 in index_to_keys:
                arr = y.squeeze(1).detach().cpu().numpy()
                paired = np.stack([arr[:n_pairs], arr[n_pairs:]], axis=1)
                for key in index_to_keys[step+1]:
                    collected[key].append(paired.copy())

        done += n_pairs
        bar.update(2*n_pairs)
        del z, y
        gc.collect()

    bar.close()
    return {k: np.concatenate(v, axis=0) for k, v in collected.items()}


def price_future_rows(rows, samples_by_am, price_col="model_price_future"):
    out = rows.copy().reset_index(drop=True)
    price = np.empty(len(out))

    for (am, T), idx in out.groupby(["am_settlement", "T"]).groups.items():
        pos = np.asarray(list(idx), dtype=int)
        g = out.loc[idx]
        samples = samples_by_am[int(am)][tkey(T)].reshape(-1)
        payoff = empirical_payoff_mean(
            samples,
            g["K"].to_numpy(float) / g["F"].to_numpy(float),
            g["cp_flag"].to_numpy(),
        )
        price[pos] = g["D"].to_numpy(float) * g["F"].to_numpy(float) * payoff

    out[price_col] = price
    out["pricing_error"] = price - out["mid"].to_numpy(float)
    out["inside_spread"] = out[price_col].between(out["bid"], out["ask"])
    return out


def evaluate_frozen_on_rows(model, model_date, target_date, rows, paths, seed):
    model_date, target_date = pd.Timestamp(model_date), pd.Timestamp(target_date)
    elapsed = (target_date-model_date).days / 365.0

    samples_by_am = {}
    for am in sorted(rows["am_settlement"].unique()):
        g = rows.loc[rows["am_settlement"] == am]
        samples_by_am[int(am)] = simulate_frozen_to_date(
            model,
            curve_book.get(model_date, int(am)),
            curve_book.get(target_date, int(am)),
            elapsed,
            g["T"].unique(),
            paths,
            FINAL_PATH_CHUNK,
            seed + int(am),
        )

    out = price_future_rows(rows, samples_by_am)
    del samples_by_am
    gc.collect()
    return out


def evaluate_transition(model, date, next_date, same_day_full, paths=FINAL_MC_PATHS, seed=SEED+8000):
    date, next_date = pd.Timestamp(date), pd.Timestamp(next_date)
    pairs = evaluation_pairs.loc[
        (evaluation_pairs["date"] == date) & (evaluation_pairs["next_date"] == next_date)
    ].copy().reset_index(drop=True)

    elapsed = (next_date-date).days / 365.0
    samples_by_am = {}
    for am in sorted(pairs["am_settlement"].unique()):
        g = pairs.loc[pairs["am_settlement"] == am]
        samples_by_am[int(am)] = simulate_frozen_to_date(
            model,
            curve_book.get(date, int(am)),
            curve_book.get(next_date, int(am)),
            elapsed,
            g["T_next"].unique(),
            paths,
            FINAL_PATH_CHUNK,
            seed + int(am),
        )

    out = pairs.copy()
    price = np.empty(len(out))
    for (am, T), idx in out.groupby(["am_settlement", "T_next"]).groups.items():
        pos = np.asarray(list(idx), dtype=int)
        g = out.loc[idx]
        samples = samples_by_am[int(am)][tkey(T)].reshape(-1)
        payoff = empirical_payoff_mean(
            samples,
            g["K"].to_numpy(float) / g["F_next"].to_numpy(float),
            g["cp_flag"].to_numpy(),
        )
        price[pos] = g["D_next"].to_numpy(float) * g["F_next"].to_numpy(float) * payoff

    out["model_price_next"] = price
    out["next_day_pricing_error"] = price - out["mid_next"].to_numpy(float)
    out["next_day_inside_bid_ask"] = out["model_price_next"].between(out["bid_next"], out["ask_next"])

    delta_map = same_day_full[
        ["optionid", "neural_lv_delta", "delta_bump_instability"]
    ].drop_duplicates("optionid")
    out = out.merge(delta_map, on="optionid", how="left", validate="many_to_one")
    out["dP_market"] = out["mid_next"] - out["mid"]
    out["dS"] = out["S_next"] - out["S"]
    out["hedge_error"] = out["dP_market"] - out["neural_lv_delta"]*out["dS"]

    pe = out["next_day_pricing_error"].to_numpy(float)
    he_mask = out["hedge_error"].notna()
    he = out.loc[he_mask, "hedge_error"].to_numpy(float)
    hp = np.maximum(out.loc[he_mask, "mid"].to_numpy(float), 1e-12)

    summary = {
        "date": date,
        "next_date": next_date,
        "n_pairs": len(out),
        "pricing_MSE": float(np.mean(pe**2)),
        "pricing_RMSE": float(np.sqrt(np.mean(pe**2))),
        "pricing_MAE": float(np.mean(np.abs(pe))),
        "pricing_Rel_MAE_pct": float(
            np.mean(np.abs(pe)/np.maximum(out["mid_next"].to_numpy(float), 1e-12))*100
        ),
        "pricing_inside_bid_ask": float(out["next_day_inside_bid_ask"].mean()),
        "hedge_MSE": float(np.mean(he**2)),
        "hedge_RMSE": float(np.sqrt(np.mean(he**2))),
        "hedge_MAE": float(np.mean(np.abs(he))),
        "hedge_Rel_MAE_pct": float(np.mean(np.abs(he)/hp)*100),
        "hedge_bias": float(np.mean(he)),
    }

    del samples_by_am
    gc.collect()
    return summary, out


def bs_delta_unit_test():
    class ConstantVol(nn.Module):
        def __init__(self, sigma):
            super().__init__()
            self.sigma = float(sigma)
        def forward(self, t, x):
            return torch.full_like(x, self.sigma)

    S, r, q, T, K, sig = 5000.0, 0.04, 0.015, 0.5, 5100.0, 0.20
    F = S*math.exp((r-q)*T)
    D = math.exp(-r*T)

    samples = simulate_same_day_x(
        ConstantVol(sig).to(DEVICE), [T], 30_000, 10_000, SEED+6600, progress=False
    )
    row = pd.DataFrame([{"S": S, "F": F, "D": D, "K": K, "T": T, "cp_flag": "C"}])
    mc = float(delta_rows_from_terminal(row, samples)["neural_lv_delta"].iloc[0])

    d1 = (math.log(S/K)+(r-q+0.5*sig*sig)*T)/(sig*math.sqrt(T))
    exact = math.exp(-q*T)*float(ndtr(d1))
    err = abs(mc-exact)

    print(f"BS delta unit test: MC={mc:.5f}, analytic={exact:.5f}, abs error={err:.5f}")
    if err > 0.03:
        warnings.warn("Delta unit test is noisier than expected.")
    return err


def clock_unit_test():
    class ToyCurve:
        am_settlement = 0
        def __init__(self, spot, carry):
            self.spot, self.carry = float(spot), float(carry)
        def forward(self, t):
            return self.spot*np.exp(self.carry*np.asarray(t, float))

    class Recorder(nn.Module):
        def __init__(self):
            super().__init__()
            self.times = []
        def forward(self, t, x):
            self.times.append(float(t.reshape(-1)[0]))
            return torch.full_like(x, 0.2)

    elapsed = 0.10
    rec = Recorder().to(DEVICE)
    simulate_frozen_to_date(
        rec, ToyCurve(100, 0.02), ToyCurve(101, 0.02),
        elapsed, [0.08], 4, 4, SEED+6700, progress=False
    )
    err = abs(rec.times[0] - elapsed)
    print(f"Clock unit test error={err:.2e}")
    if err > 1e-6:
        raise RuntimeError("Frozen model clock restarted incorrectly.")
    return err


evaluation_pairs = build_evaluation_pairs(quotes_all)


## 8. March 1 anchor — load v6 winner and re-evaluate at 100k paths

The v6 raw-loss maturity-head checkpoint is now the locked anchor. Its March 1 20% subset is reported as a **validation** result, not a pristine test result, because v6 used it to select the final loss/architecture.


In [8]:
def load_v6_anchor(ctx):
    path = locate_v6_anchor(ctx.date)
    if path is None:
        raise FileNotFoundError(
            "Completed v6 raw-loss maturity-head checkpoint not found. "
            "Expected it under neural_lv_v6_cache."
        )

    payload = safe_torch_load(path)
    model = MaturityHeadNeuralLocalVol(ctx.maturity_boundaries).to(DEVICE)
    hedge = IntervalHedgeBank(ctx.maturity_boundaries, len(ctx.contracts)).to(DEVICE)

    model_state = payload.get("final_model_state") or payload.get("model_state")
    hedge_state = payload.get("final_hedge_state") or payload.get("hedge_state")
    model.load_state_dict(model_state, strict=True)
    hedge.load_state_dict(hedge_state, strict=True)

    model.eval(); hedge.eval()
    set_requires_grad(model, False)
    set_requires_grad(hedge, False)
    print("Loaded v6 anchor:", path)
    return model, hedge, path


delta_unit_test_error = bs_delta_unit_test()
clock_unit_test_error = clock_unit_test()

anchor_model, anchor_hedge, anchor_checkpoint = load_v6_anchor(anchor_ctx)
anchor_hierarchy, anchor_full, anchor_train_eval, anchor_validation_eval, anchor_otm_eval = evaluate_same_day(
    anchor_model, ANCHOR_DATE, anchor_train, anchor_holdout,
    paths=FINAL_MC_PATHS, seed=SEED+7000
)
anchor_cv_ratio = cv_variance_ratio(anchor_model, anchor_hedge, anchor_ctx)

print("\nMarch 1 raw-loss maturity-head evaluation (100k paths)")
print(anchor_hierarchy.to_string())
print("Median delta bump instability:", float(anchor_full["delta_bump_instability"].median()))
print("CV median variance ratio:", anchor_cv_ratio)

anchor_hierarchy.to_csv(CACHE_DIR / "anchor_20240301_same_day.csv")
anchor_validation_eval.to_csv(CACHE_DIR / "anchor_20240301_validation_contracts.csv", index=False)


BS delta unit test: MC=0.50156, analytic=0.50382, abs error=0.00226
Clock unit test error=1.49e-09
Loaded v6 anchor: neural_lv_v6_cache\real_20240301_split80_20_ablation_rawloss_maturity_heads_resume.pt


2024-03-01 holdout IV: 100%|██████████| 199/199 [00:00<00:00, 12133.72opt/s]



March 1 raw-loss maturity-head evaluation (100k paths)
                           n        MSE      RMSE       MAE  Rel_MAE_pct  inside_bid_ask   IV_RMSE
selected_train_80      798.0   2.141677  1.463447  1.015404     4.147340        0.165414  0.003627
selected_holdout_20    199.0   2.532283  1.591315  1.100990     4.196542        0.160804  0.003581
full_OTM_strip        4304.0   2.420079  1.555660  1.039272     3.466270        0.158457  0.003171
complete_clean_chain  9046.0  33.186472  5.760770  4.178395     2.779205        0.129560       NaN
Median delta bump instability: 3.349583724049365e-05
CV median variance ratio: 0.04133220762014389


## 9. Main experiment — rolling recalibration and future-date tests

CPU profile runs the first five consecutive transitions from March 1. GPU/HPC profile runs every available transition in the loaded month.

For each later calibration date:

1. rebuild that day's deterministic 80/20 split;
2. warm-start trunk and shifted maturity heads from the previous trading day's model;
3. initialise a fresh control-variate hedge bank;
4. train for at most 120 nominal epochs with training-only early stopping;
5. evaluate at 100k paths on the same-day train/hold-out/full surfaces;
6. compare that day's hold-out with the **frozen March 1 anchor**;
7. freeze the newly calibrated model and evaluate next-day pricing + delta hedging.

The frozen-anchor comparison measures whether daily recalibration is genuinely adding value.


In [9]:
anchor_index = trading_dates.index(ANCHOR_DATE)
available_transitions = len(trading_dates) - anchor_index - 1
n_transitions = available_transitions if MAX_TRANSITIONS is None else min(MAX_TRANSITIONS, available_transitions)
sequence = trading_dates[anchor_index:anchor_index+n_transitions+1]

print("Rolling sequence:", " -> ".join(str(pd.Timestamp(d).date()) for d in sequence))

daily_records = []
transition_records = []
same_day_exports = []
transition_exports = []

# March 1: validation only, because it participated in v6 model selection.
a_hold = anchor_hierarchy.loc["selected_holdout_20"]
daily_records.append({
    "date": ANCHOR_DATE,
    "role": "anchor_validation",
    "trained_epochs": 0,
    "stop_reason": "loaded_v6_anchor",
    "train_RMSE": float(anchor_hierarchy.loc["selected_train_80", "RMSE"]),
    "holdout_RMSE": float(a_hold["RMSE"]),
    "holdout_MAE": float(a_hold["MAE"]),
    "holdout_IV_RMSE": float(a_hold["IV_RMSE"]),
    "holdout_inside_bid_ask": float(a_hold["inside_bid_ask"]),
    "full_OTM_RMSE": float(anchor_hierarchy.loc["full_OTM_strip", "RMSE"]),
    "frozen_anchor_holdout_RMSE": float(a_hold["RMSE"]),
})
same_day_exports.append(anchor_validation_eval.assign(date=ANCHOR_DATE, split="anchor_validation"))

# First transition uses the already-calibrated anchor.
if n_transitions > 0:
    s, t = sequence[0], sequence[1]
    ts, tr = evaluate_transition(
        anchor_model, s, t, anchor_full,
        paths=FINAL_MC_PATHS, seed=SEED+8000
    )
    transition_records.append(ts)
    transition_exports.append(tr)

previous_model = anchor_model
previous_date = ANCHOR_DATE

# The final sequence date is only the target of the last transition.
for i, date in enumerate(sequence[1:-1], start=1):
    date = pd.Timestamp(date)
    next_date = pd.Timestamp(sequence[i+1])

    print("\n" + "="*78)
    print("ROLLING CALIBRATION:", date.date(), "->", next_date.date())

    selected, train, holdout = daily_split(date)
    ctx = build_calibration_context(date, train)
    cfg = TrainConfig(seed_offset=1000*i)

    run_tag = f"rolling_{date:%Y%m%d}_raw_heads"
    model, hedge, hist, rerank, stop_reason = train_rolling_day(
        ctx, cfg, run_tag,
        previous_model=previous_model,
        previous_date=previous_date,
    )

    hierarchy, full_eval, train_eval, holdout_eval, otm_eval = evaluate_same_day(
        model, date, train, holdout,
        paths=FINAL_MC_PATHS, seed=SEED+7000+i*100
    )

    # No-recalibration comparator: frozen March 1 model prices this day's exact hold-out.
    target_rows = quotes_all.loc[
        (quotes_all["date"] == date)
        & (quotes_all["optionid"].astype(int).isin(selected_quote_ids(holdout)))
    ].copy().reset_index(drop=True)

    frozen_eval = evaluate_frozen_on_rows(
        anchor_model, ANCHOR_DATE, date, target_rows,
        FINAL_MC_PATHS, SEED+12000+i*100
    )
    frozen_summary = price_summary(frozen_eval)

    h = hierarchy.loc["selected_holdout_20"]
    daily_records.append({
        "date": date,
        "role": "clean_future_holdout",
        "trained_epochs": int(hist["epoch"].max()),
        "stop_reason": stop_reason,
        "train_RMSE": float(hierarchy.loc["selected_train_80", "RMSE"]),
        "holdout_RMSE": float(h["RMSE"]),
        "holdout_MAE": float(h["MAE"]),
        "holdout_IV_RMSE": float(h["IV_RMSE"]),
        "holdout_inside_bid_ask": float(h["inside_bid_ask"]),
        "full_OTM_RMSE": float(hierarchy.loc["full_OTM_strip", "RMSE"]),
        "frozen_anchor_holdout_RMSE": float(frozen_summary["RMSE"]),
    })
    same_day_exports.append(holdout_eval.assign(date=date, split="clean_future_holdout"))

    ts, tr = evaluate_transition(
        model, date, next_date, full_eval,
        paths=FINAL_MC_PATHS, seed=SEED+8000+i*100
    )
    transition_records.append(ts)
    transition_exports.append(tr)

    previous_model = model
    previous_date = date

    del hedge, ctx, full_eval, train_eval, holdout_eval, otm_eval, frozen_eval
    gc.collect()


daily_summary = pd.DataFrame(daily_records)
transition_summary = pd.DataFrame(transition_records)

print("\nDAILY SAME-DAY SUMMARY")
print(daily_summary.to_string(index=False))

print("\nNEXT-DAY PRICING / HEDGING SUMMARY")
print(transition_summary.to_string(index=False))

daily_summary.to_csv(CACHE_DIR / "v7_daily_summary.csv", index=False)
transition_summary.to_csv(CACHE_DIR / "v7_transition_summary.csv", index=False)
pd.concat(same_day_exports, ignore_index=True).to_csv(
    CACHE_DIR / "v7_holdout_contract_results.csv", index=False
)
pd.concat(transition_exports, ignore_index=True).to_csv(
    CACHE_DIR / "v7_next_day_pricing_hedging_contracts.csv", index=False
)


Rolling sequence: 2024-03-01 -> 2024-03-04 -> 2024-03-05 -> 2024-03-06 -> 2024-03-07 -> 2024-03-08


Future-date MC AM=1: 100%|██████████| 100000/100000 [00:05<00:00, 17311.37path/s]



ROLLING CALIBRATION: 2024-03-04 -> 2024-03-05
Warm-start 2024-03-01 -> 2024-03-04: 36 heads, median maturity mismatch=0.00000y


v7 rolling_20240304_raw_heads:  82%|████████▏ | 98/120 [1:00:51<13:39, 37.26s/epoch, phase=sde, loss=1.47e-07, RMSE=2.574, patience=4/4]


Early stopping: early_stop: no >= 0.20% monitor improvement for 4 checks | epoch 99


...


ROLLING CALIBRATION: 2024-03-07 -> 2024-03-08
Warm-start 2024-03-06 -> 2024-03-07: 36 heads, median maturity mismatch=0.00000y


Independent checkpoint rerank: 100%|██████████| 3/3 [00:17<00:00,  5.89s/model]


Selected epoch 109: rerank raw RMSE=1.7907


Future-date MC AM=1: 100%|██████████| 100000/100000 [00:09<00:00, 10835.58path/s]



DAILY SAME-DAY SUMMARY
      date                 role  trained_epochs                                              stop_reason  train_RMSE  holdout_RMSE  holdout_MAE  holdout_IV_RMSE  holdout_inside_bid_ask  full_OTM_RMSE  frozen_anchor_holdout_RMSE
2024-03-01    anchor_validation               0                                         loaded_v6_anchor    1.463447      1.591315     1.100990         0.003581                0.160804       1.555660                    1.591315
2024-03-04 clean_future_holdout              99 early_stop: no >= 0.20% monitor improvement for 4 checks    1.714258      1.756197     1.136032         0.002923                0.140000       1.746795                    2.510972
2024-03-05 clean_future_holdout              89 early_stop: no >= 0.20% monitor improvement for 4 checks    1.880335      2.075390     1.462629         0.003610                0.140000       1.964765                    4.137995
2024-03-06 clean_future_holdout             119 early_stop: no >

## 10. Compact final NLV summary


In [10]:
future_days = daily_summary.loc[daily_summary["role"] == "clean_future_holdout"].copy()

final_summary = {
    "model": "Neural LV",
    "architecture": "shared_4x50_trunk_plus_maturity_heads",
    "loss": "maturity_balanced_raw_price_MSE_spot_scaled",
    "anchor_date": str(ANCHOR_DATE.date()),
    "compute_profile": COMPUTE_PROFILE,
    "n_transitions": int(len(transition_summary)),
    "anchor_validation_RMSE_100k": float(
        daily_summary.loc[daily_summary["role"] == "anchor_validation", "holdout_RMSE"].iloc[0]
    ),
    "mean_future_holdout_RMSE": float(future_days["holdout_RMSE"].mean()) if len(future_days) else np.nan,
    "mean_frozen_anchor_same_contract_RMSE": float(
        future_days["frozen_anchor_holdout_RMSE"].mean()
    ) if len(future_days) else np.nan,
    "mean_next_day_pricing_RMSE": float(transition_summary["pricing_RMSE"].mean()),
    "mean_next_day_pricing_MAE": float(transition_summary["pricing_MAE"].mean()),
    "mean_one_day_hedge_RMSE": float(transition_summary["hedge_RMSE"].mean()),
    "mean_one_day_hedge_MAE": float(transition_summary["hedge_MAE"].mean()),
    "mean_one_day_hedge_bias": float(transition_summary["hedge_bias"].mean()),
    "anchor_cv_median_variance_ratio": float(anchor_cv_ratio),
    "anchor_delta_bump_instability": float(anchor_full["delta_bump_instability"].median()),
    "BS_delta_unit_test_abs_error": float(delta_unit_test_error),
    "clock_unit_test_abs_error": float(clock_unit_test_error),
}

print(json.dumps(final_summary, indent=2))
pd.DataFrame([final_summary]).to_csv(CACHE_DIR / "v7_final_nlv_summary.csv", index=False)
print("\nSaved v7 outputs under:", CACHE_DIR.resolve())


{
  "model": "Neural LV",
  "architecture": "shared_4x50_trunk_plus_maturity_heads",
  "loss": "maturity_balanced_raw_price_MSE_spot_scaled",
  "anchor_date": "2024-03-01",
  "compute_profile": "cpu",
  "n_transitions": 5,
  "anchor_validation_RMSE_100k": 1.5913149426107687,
  "mean_future_holdout_RMSE": 1.9348796154930326,
  "mean_frozen_anchor_same_contract_RMSE": 4.68905262584575,
  "mean_next_day_pricing_RMSE": 7.3208644983574285,
  "mean_next_day_pricing_MAE": 5.9920110829964015,
  "mean_one_day_hedge_RMSE": 2.3446504539563007,
  "mean_one_day_hedge_MAE": 2.04891424028841,
  "mean_one_day_hedge_bias": 1.1094920474465344,
  "anchor_cv_median_variance_ratio": 0.04133220762014389,
  "anchor_delta_bump_instability": 3.349583724049365e-05,
  "BS_delta_unit_test_abs_error": 0.00225702690985774,
  "clock_unit_test_abs_error": 1.4901161138336505e-09
}

Saved v7 outputs under: C:\Users\Startklar\ICL\NDEs\Codes\NSDE\neural_lv_v7_cache


## 11. OPTIONAL — three-seed rolling robustness

This is deliberately separate from the main run. It repeats the **first future-date rolling recalibration** three times from the same locked March 1 anchor, changing only the stochastic seed.

That tests sensitivity to Monte Carlo paths, minibatch sampling and SGD noise under the actual v7 rolling procedure. It is cheaper and cleaner than retraining the whole anchor three times on CPU.

If GPU/HPC later becomes available, the stronger additional robustness experiment is three genuinely independent cold-start March 1 calibrations; that is not required for the normal v7 run.


In [ ]:
if not RUN_SEED_ROBUSTNESS:
    print("Seed robustness skipped. Set RUN_SEED_ROBUSTNESS=True when desired.")
elif len(sequence) < 2:
    print("No future date is available for rolling seed robustness.")
else:
    seed_date = pd.Timestamp(sequence[1])
    _, seed_train, seed_holdout = daily_split(seed_date)
    seed_ctx = build_calibration_context(seed_date, seed_train)
    seed_rows = []

    for j, offset in enumerate([31_000, 32_000, 33_000], start=1):
        cfg = TrainConfig(seed_offset=offset)
        tag = f"seed_robust_{seed_date:%Y%m%d}_{j}"
        m, h, hist, rerank, reason = train_rolling_day(
            seed_ctx, cfg, tag,
            previous_model=anchor_model,
            previous_date=ANCHOR_DATE,
        )
        hierarchy, *_ = evaluate_same_day(
            m, seed_date, seed_train, seed_holdout,
            paths=FINAL_MC_PATHS, seed=SEED+40_000+j*100
        )
        seed_rows.append({
            "seed_run": j,
            "heldout_RMSE": float(hierarchy.loc["selected_holdout_20", "RMSE"]),
            "heldout_IV_RMSE": float(hierarchy.loc["selected_holdout_20", "IV_RMSE"]),
            "trained_epochs": int(hist["epoch"].max()),
            "stop_reason": reason,
        })
        del m, h
        gc.collect()

    seed_robustness = pd.DataFrame(seed_rows)
    print(seed_robustness.to_string(index=False))
    seed_robustness.to_csv(CACHE_DIR / "v7_rolling_seed_robustness.csv", index=False)


### v7 stopping point

If the future-date hold-outs, rolling next-day pricing and hedging remain stable, Neural LV is frozen after this notebook.

The next modelling stage is Neural SLV. BS-IVF, classical LV, Heston, NLV and NSLV should then be compared in a separate thin benchmark notebook using common contract sets and common pricing/hedging metrics.
